# 🚀 Nanochat Model Conversion & Upload

This notebook converts nanochat checkpoints to HuggingFace format and optionally to GGUF format for use with llama.cpp.

**Features:**
- Convert nanochat SFT/RL checkpoints to HuggingFace transformers format
- Optionally convert to GGUF format (for llama.cpp, ollama, etc.)
- Upload converted models to HuggingFace Hub

**Requirements:**
- Google Colab with GPU runtime (recommended: T4 or higher)
- HuggingFace account and API token

## 📋 Configuration

Set your configuration here before running the notebook.

In [ ]:
# ============================================================================
# CONFIGURATION - Edit these values
# ============================================================================
import os

# HuggingFace settings
HF_USERNAME = "pankajmathur"  # Your HuggingFace username
HF_TOKEN = ""  # Your HuggingFace token (or set via huggingface-cli login)

# Source model repository (where to download checkpoint from)
SOURCE_HF_REPO = "pankajmathur/nanochat-d34-sft"

# Output repository names
HF_REPO_NAME = "nanochat-d34-sft-hf"  # For HuggingFace format
GGUF_REPO_NAME = "nanochat-d34-sft-GGUF"  # For GGUF format

# Conversion options
CONVERT_TO_HF = False  # Convert to HuggingFace format
CONVERT_TO_GGUF = True  # Convert to GGUF (uses custom converter, bypasses llama.cpp limitation)

# GGUF settings
# Base dtype for initial GGUF (f16 recommended)
GGUF_BASE_DTYPE = "f16"
# Additional quantizations to create (requires llama.cpp llama-quantize binary)
# Options: q8_0, q4_0, q4_1, q5_0, q5_1, q2_K, q3_K_S, q3_K_M, q4_K_S, q4_K_M, q5_K_S, q5_K_M, q6_K
GGUF_QUANTIZATIONS = ["q6_K", "q4_K_M"]  # Set to [] to skip quantization

# Checkpoint source type
CHECKPOINT_SOURCE = "sft"  # Options: "base", "mid", "sft", "rl"

# Local directories - auto-detect environment
# Colab uses /content, other environments use home directory
def get_work_dir():
    if os.path.exists("/content") and os.access("/content", os.W_OK):
        return "/content/nanochat_convert"  # Google Colab
    else:
        return os.path.expanduser("~/nanochat_convert")  # Lambda Labs, local, etc.

WORK_DIR = get_work_dir()
HF_OUTPUT_DIR = f"{WORK_DIR}/hf_model"
GGUF_OUTPUT_DIR = f"{WORK_DIR}/gguf_models"

print(f"📁 Working directory: {WORK_DIR}")

## 1️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch safetensors sentencepiece
!pip install -q huggingface_hub
# Install transformers from source (required for NanoChat support)
!pip install -q git+https://github.com/huggingface/transformers.git

print("✅ Base dependencies installed")

In [ ]:
# Verify installations
import os
import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 2️⃣ HuggingFace Login

In [ ]:
from huggingface_hub import login, HfApi

# Login to HuggingFace
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged in with provided token")
else:
    print("⚠️ No token provided, attempting interactive login...")
    login()

# Verify login
api = HfApi()
try:
    user_info = api.whoami()
    print(f"✅ Logged in as: {user_info['name']}")
except Exception as e:
    print(f"❌ Login verification failed: {e}")

## 3️⃣ Download Model Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
import os

# Create work directory
os.makedirs(WORK_DIR, exist_ok=True)

# Download checkpoint from HuggingFace
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoint"

print(f"📥 Downloading checkpoint from {SOURCE_HF_REPO}...")
snapshot_download(
    repo_id=SOURCE_HF_REPO,
    local_dir=CHECKPOINT_DIR,
    local_dir_use_symlinks=False,
)

print(f"✅ Checkpoint downloaded to: {CHECKPOINT_DIR}")

# List downloaded files
print("\n📁 Downloaded files:")
for root, dirs, files in os.walk(CHECKPOINT_DIR):
    level = root.replace(CHECKPOINT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit to 10 files per directory
        print(f"{subindent}{file}")
    if len(files) > 10:
        print(f"{subindent}... and {len(files) - 10} more files")

## 4️⃣ Convert to HuggingFace Format

In [ ]:
import os
import json
import glob
import torch
import gc
from pathlib import Path

if not CONVERT_TO_HF:
    print("⏭️ Skipping HuggingFace conversion (disabled in config)")
else:
    os.makedirs(HF_OUTPUT_DIR, exist_ok=True)
    
    # Determine checkpoint directory based on source type
    checkpoint_type_dirs = {
        "base": "base_checkpoints",
        "mid": "mid_checkpoints",
        "sft": "chatsft_checkpoints",
        "rl": "chatrl_checkpoints",
    }
    
    # Find the checkpoint directory
    checkpoint_subdir = checkpoint_type_dirs.get(CHECKPOINT_SOURCE, "chatsft_checkpoints")
    possible_dirs = [
        f"{CHECKPOINT_DIR}/{checkpoint_subdir}/d34",
        f"{CHECKPOINT_DIR}/{checkpoint_subdir}",
        CHECKPOINT_DIR,
    ]
    
    INPUT_DIR = None
    for d in possible_dirs:
        if os.path.exists(d) and (glob.glob(f"{d}/model_*.pt") or glob.glob(f"{d}/*.safetensors")):
            INPUT_DIR = d
            break
    
    if INPUT_DIR is None:
        # Try to find any model files
        model_files = glob.glob(f"{CHECKPOINT_DIR}/**/model_*.pt", recursive=True)
        if model_files:
            INPUT_DIR = os.path.dirname(model_files[0])
        else:
            raise FileNotFoundError(f"No checkpoint found in {CHECKPOINT_DIR}")
    
    print(f"📁 Using checkpoint directory: {INPUT_DIR}")
    
    # Find tokenizer directory
    TOKENIZER_DIR = f"{CHECKPOINT_DIR}/tokenizer"
    if not os.path.exists(TOKENIZER_DIR):
        TOKENIZER_DIR = CHECKPOINT_DIR
    
    print(f"📁 Using tokenizer directory: {TOKENIZER_DIR}")

In [ ]:
# Conversion functions

def infer_kv_heads(hidden_size, num_attention_heads, state_dict):
    """Infer number of key-value heads from checkpoint weights."""
    key_weight = state_dict.get("transformer.h.0.attn.c_k.weight")
    if key_weight is None:
        return num_attention_heads
    rows = key_weight.shape[0]
    head_dim = hidden_size // num_attention_heads
    if rows % head_dim != 0:
        return num_attention_heads
    inferred = rows // head_dim
    print(f"Inferred {inferred} key_value heads from checkpoint")
    return max(inferred, 1)


def convert_layer(old_prefix, new_prefix):
    """Map nanochat layer keys to HuggingFace transformers layer keys."""
    return {
        f"{old_prefix}.attn.c_q.weight": f"{new_prefix}.self_attn.q_proj.weight",
        f"{old_prefix}.attn.c_k.weight": f"{new_prefix}.self_attn.k_proj.weight",
        f"{old_prefix}.attn.c_v.weight": f"{new_prefix}.self_attn.v_proj.weight",
        f"{old_prefix}.attn.c_proj.weight": f"{new_prefix}.self_attn.o_proj.weight",
        f"{old_prefix}.mlp.c_fc.weight": f"{new_prefix}.mlp.fc1.weight",
        f"{old_prefix}.mlp.c_proj.weight": f"{new_prefix}.mlp.fc2.weight",
    }


def load_config_from_checkpoint(input_path):
    """Load config from checkpoint directory."""
    input_path = Path(input_path)
    
    # Try to find meta_*.json first
    meta_files = list(input_path.glob("meta_*.json"))
    
    if meta_files:
        meta_file = meta_files[0]
        print(f"Loading config from {meta_file.name}")
        with open(meta_file, "r") as f:
            meta_config = json.load(f)
        
        if "model_config" in meta_config:
            model_config = meta_config["model_config"]
        else:
            model_config = meta_config
        
        config_kwargs = {
            "vocab_size": model_config.get("vocab_size", 50304),
            "hidden_size": model_config.get("n_embd", 768),
            "num_hidden_layers": model_config.get("n_layer", 12),
            "num_attention_heads": model_config.get("n_head", 6),
            "num_key_value_heads": model_config.get("n_kv_head"),
            "max_position_embeddings": model_config.get("sequence_len", 2048),
            "intermediate_size": model_config.get("intermediate_size", model_config.get("n_embd", 768) * 4),
        }
        
        # Try to load additional config from config.json
        config_file = input_path / "config.json"
        if config_file.exists():
            with open(config_file, "r") as f:
                extra_config = json.load(f)
            for key in ["hidden_act", "attention_dropout", "rms_norm_eps", 
                       "initializer_range", "logits_soft_cap", "attention_bias",
                       "bos_token_id", "eos_token_id", "pad_token_id", "rope_theta"]:
                if key in extra_config:
                    config_kwargs[key] = extra_config[key]
        
        return config_kwargs
    
    # Fallback to config.json
    config_file = input_path / "config.json"
    if config_file.exists():
        with open(config_file, "r") as f:
            return json.load(f)
    
    raise ValueError(f"No config file found in {input_path}")

print("✅ Conversion functions defined")

In [ ]:
if CONVERT_TO_HF:
    try:
        from transformers import NanoChatConfig, NanoChatForCausalLM
        HAS_NANOCHAT = True
        print("✅ transformers with NanoChat support found")
    except ImportError:
        HAS_NANOCHAT = False
        print("⚠️ transformers with NanoChat support not found")
        print("Installing latest transformers...")
        !pip install -q git+https://github.com/huggingface/transformers.git
        from transformers import NanoChatConfig, NanoChatForCausalLM
        print("✅ Installed transformers with NanoChat support")
    
    # Load config
    config_kwargs = load_config_from_checkpoint(INPUT_DIR)
    print(f"\nConfig: hidden_size={config_kwargs.get('hidden_size')}, num_layers={config_kwargs.get('num_hidden_layers')}")
    
    # Find checkpoint file
    checkpoint_files = sorted(glob.glob(f"{INPUT_DIR}/model_*.pt"))
    if checkpoint_files:
        checkpoint_path = checkpoint_files[-1]  # Use the latest
    else:
        checkpoint_path = f"{INPUT_DIR}/pytorch_model.bin"
    
    print(f"📥 Loading checkpoint from: {checkpoint_path}")
    old_state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    
    # Handle torch.compile prefix
    old_state = {k.removeprefix("_orig_mod."): v for k, v in old_state.items()}
    
    # Convert to bfloat16
    for key in old_state:
        if old_state[key].dtype == torch.float32:
            old_state[key] = old_state[key].to(torch.bfloat16)
    
    print(f"✅ Loaded {len(old_state)} tensors")

In [ ]:
if CONVERT_TO_HF:
    # Create config
    config = NanoChatConfig(**config_kwargs)
    
    # Infer KV heads
    inferred_kv = infer_kv_heads(config.hidden_size, config.num_attention_heads, old_state)
    config.num_key_value_heads = inferred_kv
    
    # Convert state dict
    print("🔄 Converting model...")
    state_dict = {}
    rename_map = {}
    
    def assign(old_key, new_key):
        tensor = old_state.get(old_key)
        if tensor is not None:
            state_dict[new_key] = tensor.clone()
            rename_map[old_key] = new_key
    
    # Convert embeddings
    assign("transformer.wte.weight", "model.embed_tokens.weight")
    assign("lm_head.weight", "lm_head.weight")
    
    # Convert layers
    for layer_idx in range(config.num_hidden_layers):
        old_prefix = f"transformer.h.{layer_idx}"
        new_prefix = f"model.layers.{layer_idx}"
        mapping = convert_layer(old_prefix, new_prefix)
        for old_key, new_key in mapping.items():
            assign(old_key, new_key)
    
    missing = [key for key in old_state.keys() if key not in rename_map]
    if missing:
        print(f"⚠️ Skipped {len(missing)} legacy entries")
    
    del old_state
    gc.collect()
    
    print(f"✅ Converted {len(state_dict)} tensors")

In [ ]:
if CONVERT_TO_HF:
    # Create and load model
    print("📦 Creating HuggingFace model...")
    config.tie_word_embeddings = False
    
    with torch.device("meta"):
        model = NanoChatForCausalLM(config)
    
    model.load_state_dict(state_dict, strict=True, assign=True)
    print("✅ Model loaded")
    
    # Save model
    print(f"💾 Saving model to {HF_OUTPUT_DIR}...")
    model.save_pretrained(HF_OUTPUT_DIR, safe_serialization=True)
    
    del state_dict, model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    print("✅ Model saved")

In [ ]:
if CONVERT_TO_HF:
    # Convert tokenizer
    print("🔄 Converting tokenizer...")
    
    tokenizer_pkl = f"{TOKENIZER_DIR}/tokenizer.pkl"
    if os.path.exists(tokenizer_pkl):
        try:
            import pickle
            from transformers.integrations.tiktoken import convert_tiktoken_to_fast
            
            with open(tokenizer_pkl, "rb") as f:
                tok_pkl = pickle.load(f)
            convert_tiktoken_to_fast(tok_pkl, HF_OUTPUT_DIR)
            print("✅ Tokenizer converted")
        except Exception as e:
            print(f"⚠️ Failed to convert tokenizer: {e}")
            # Copy existing tokenizer files
            import shutil
            for f in ["tokenizer.json", "tokenizer_config.json"]:
                src = f"{TOKENIZER_DIR}/{f}"
                if os.path.exists(src):
                    shutil.copy(src, HF_OUTPUT_DIR)
    else:
        print("⚠️ No tokenizer.pkl found, looking for tokenizer files...")
        import shutil
        for f in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
            src = f"{TOKENIZER_DIR}/{f}"
            if os.path.exists(src):
                shutil.copy(src, HF_OUTPUT_DIR)
                print(f"  Copied {f}")
    
    print(f"\n✅ HuggingFace model saved to: {HF_OUTPUT_DIR}")
    !ls -la {HF_OUTPUT_DIR}

## 5️⃣ Convert to GGUF Format

This uses a **custom GGUF converter** that bypasses llama.cpp's `convert_hf_to_gguf.py` (which doesn't support NanoChat).

The converter maps NanoChat to LLaMA-compatible GGUF format for use with llama.cpp, Ollama, LM Studio, etc.

In [ ]:
#@title GGUF Converter (hidden - defines convert_nanochat_to_gguf function)
#@markdown This cell contains the custom NanoChat→GGUF converter. Run it to define the conversion function.

import json
from pathlib import Path
from typing import Any
import numpy as np

def convert_nanochat_to_gguf(input_dir: str, output_file: str, dtype: str = "f16"):
    """
    Convert NanoChat HuggingFace model to GGUF format.
    Maps NanoChat architecture to LLaMA-compatible GGUF format.
    """
    import gguf
    from gguf import GGUFWriter, GGMLQuantizationType
    
    input_path = Path(input_dir)
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Load config
    with open(input_path / "config.json") as f:
        config = json.load(f)
    
    # Load weights
    state_dict = {}
    safetensors_files = list(input_path.glob("*.safetensors"))
    pytorch_files = list(input_path.glob("pytorch_model*.bin"))
    
    if safetensors_files:
        from safetensors.torch import load_file
        for sf in safetensors_files:
            state_dict.update(load_file(sf))
    elif pytorch_files:
        for pf in pytorch_files:
            state_dict.update(torch.load(pf, map_location="cpu", weights_only=True))
    
    print(f"Loaded {len(state_dict)} tensors from {input_path}")
    
    # Extract config
    vocab_size = config.get("vocab_size", 50304)
    hidden_size = config.get("hidden_size", 768)
    num_layers = config.get("num_hidden_layers", 12)
    num_heads = config.get("num_attention_heads", 6)
    num_kv_heads = config.get("num_key_value_heads", num_heads)
    max_seq_len = config.get("max_position_embeddings", 2048)
    intermediate_size = config.get("intermediate_size", hidden_size * 4)
    rms_norm_eps = config.get("rms_norm_eps", 1e-5)
    rope_theta = config.get("rope_theta", 10000.0)
    
    print(f"Model: {num_layers} layers, {hidden_size} hidden, {num_heads} heads, {vocab_size} vocab")
    
    # GGUF dtype mapping
    dtype_map = {
        "f32": GGMLQuantizationType.F32,
        "f16": GGMLQuantizationType.F16,
        "bf16": GGMLQuantizationType.BF16,
    }
    gguf_dtype = dtype_map.get(dtype, GGMLQuantizationType.F16)
    
    def to_numpy(tensor):
        if tensor.dtype == torch.bfloat16:
            tensor = tensor.to(torch.float32)
        return tensor.to(torch.float16 if dtype != "f32" else torch.float32).numpy()
    
    # Create GGUF writer with LLaMA architecture
    writer = GGUFWriter(str(output_path), arch="llama")
    
    # Metadata
    writer.add_name("nanochat")
    writer.add_context_length(max_seq_len)
    writer.add_embedding_length(hidden_size)
    writer.add_block_count(num_layers)
    writer.add_feed_forward_length(intermediate_size)
    writer.add_head_count(num_heads)
    writer.add_head_count_kv(num_kv_heads)
    writer.add_layer_norm_rms_eps(rms_norm_eps)
    writer.add_rope_freq_base(rope_theta)
    writer.add_vocab_size(vocab_size)
    writer.add_file_type(gguf_dtype)
    writer.add_bos_token_id(config.get("bos_token_id", 1))
    writer.add_eos_token_id(config.get("eos_token_id", 2))
    
    # Token embeddings
    if "model.embed_tokens.weight" in state_dict:
        writer.add_tensor("token_embd.weight", to_numpy(state_dict["model.embed_tokens.weight"]))
    
    # Output head
    if "lm_head.weight" in state_dict:
        writer.add_tensor("output.weight", to_numpy(state_dict["lm_head.weight"]))
    
    # Final norm
    if "model.norm.weight" in state_dict:
        writer.add_tensor("output_norm.weight", to_numpy(state_dict["model.norm.weight"]))
    
    # Layers
    for i in range(num_layers):
        hf = f"model.layers.{i}"
        blk = f"blk.{i}"
        
        # Attention
        for src, dst in [("q_proj", "attn_q"), ("k_proj", "attn_k"), ("v_proj", "attn_v"), ("o_proj", "attn_output")]:
            key = f"{hf}.self_attn.{src}.weight"
            if key in state_dict:
                writer.add_tensor(f"{blk}.{dst}.weight", to_numpy(state_dict[key]))
        
        # MLP
        fc1_key = f"{hf}.mlp.fc1.weight"
        if fc1_key in state_dict:
            writer.add_tensor(f"{blk}.ffn_up.weight", to_numpy(state_dict[fc1_key]))
        fc2_key = f"{hf}.mlp.fc2.weight"
        if fc2_key in state_dict:
            writer.add_tensor(f"{blk}.ffn_down.weight", to_numpy(state_dict[fc2_key]))
        
        # Also check for gate/up/down format
        for src, dst in [("gate_proj", "ffn_gate"), ("up_proj", "ffn_up"), ("down_proj", "ffn_down")]:
            key = f"{hf}.mlp.{src}.weight"
            if key in state_dict:
                writer.add_tensor(f"{blk}.{dst}.weight", to_numpy(state_dict[key]))
        
        # Layer norms
        for src, dst in [("input_layernorm", "attn_norm"), ("post_attention_layernorm", "ffn_norm")]:
            key = f"{hf}.{src}.weight"
            if key in state_dict:
                writer.add_tensor(f"{blk}.{dst}.weight", to_numpy(state_dict[key]))
    
    # Write file
    writer.write_header_to_file()
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    
    size_gb = output_path.stat().st_size / (1024**3)
    print(f"✅ GGUF saved: {output_path} ({size_gb:.2f} GB)")
    return str(output_path)

print("✅ GGUF converter function defined: convert_nanochat_to_gguf()")

In [ ]:
if not CONVERT_TO_GGUF:
    print("⏭️ Skipping GGUF conversion (disabled in config)")
else:
    # Install dependencies
    !pip install -q gguf safetensors
    
    os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)
    
    # Determine source directory
    if CONVERT_TO_HF and os.path.exists(f"{HF_OUTPUT_DIR}/config.json"):
        GGUF_SOURCE_DIR = HF_OUTPUT_DIR
        print(f"📁 Using converted HF model: {GGUF_SOURCE_DIR}")
    else:
        GGUF_SOURCE_DIR = f"{WORK_DIR}/hf_model_for_gguf"
        HF_MODEL_REPO = f"{HF_USERNAME}/{HF_REPO_NAME}"
        print(f"📥 Downloading HF model from {HF_MODEL_REPO}...")
        snapshot_download(repo_id=HF_MODEL_REPO, local_dir=GGUF_SOURCE_DIR, local_dir_use_symlinks=False)
    
    model_name = HF_REPO_NAME.replace("-hf", "").replace("_", "-")
    gguf_files = []
    
    # Step 1: Create base GGUF using the converter defined above
    base_gguf = f"{GGUF_OUTPUT_DIR}/{model_name}-{GGUF_BASE_DTYPE}.gguf"
    print(f"\n🔄 Creating base GGUF ({GGUF_BASE_DTYPE})...")
    
    try:
        convert_nanochat_to_gguf(GGUF_SOURCE_DIR, base_gguf, GGUF_BASE_DTYPE)
        gguf_files.append(base_gguf)
    except Exception as e:
        print(f"❌ Conversion failed: {e}")
        import traceback
        traceback.print_exc()
    
    # Step 2: Additional quantizations using llama-quantize (if requested)
    if os.path.exists(base_gguf) and GGUF_QUANTIZATIONS:
        print(f"\n🔧 Building llama.cpp for quantization (using CMake)...")
        LLAMA_CPP_DIR = f"{WORK_DIR}/llama.cpp"
        if not os.path.exists(LLAMA_CPP_DIR):
            !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {LLAMA_CPP_DIR}
        
        # Clear old build to avoid CMake cache issues
        import shutil
        build_dir = f"{LLAMA_CPP_DIR}/build"
        if os.path.exists(build_dir):
            shutil.rmtree(build_dir)
        
        # Build with CMake (llama.cpp no longer uses Makefile)
        # Disable CURL and other optional features we don't need for quantization
        !cd {LLAMA_CPP_DIR} && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=OFF && cmake --build build --target llama-quantize -j
        
        QUANTIZE_BIN = f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"
        if not os.path.exists(QUANTIZE_BIN):
            # Try alternative path
            QUANTIZE_BIN = f"{LLAMA_CPP_DIR}/build/llama-quantize"
        
        if os.path.exists(QUANTIZE_BIN):
            for quant in GGUF_QUANTIZATIONS:
                quant_file = f"{GGUF_OUTPUT_DIR}/{model_name}-{quant}.gguf"
                print(f"\n🔄 Quantizing to {quant}...")
                !{QUANTIZE_BIN} {base_gguf} {quant_file} {quant.upper()}
                if os.path.exists(quant_file):
                    qsize = os.path.getsize(quant_file) / (1024**3)
                    print(f"   ✅ Created: {quant_file} ({qsize:.2f} GB)")
                    gguf_files.append(quant_file)
        else:
            print("⚠️ llama-quantize build failed, skipping additional quantizations")
            print(f"   Looked for: {QUANTIZE_BIN}")
    
    print(f"\n✅ GGUF conversion complete! Created {len(gguf_files)} files:")
    !ls -lh {GGUF_OUTPUT_DIR}/*.gguf 2>/dev/null || echo "No GGUF files found"

## 6️⃣ Upload to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Upload HuggingFace format model
if CONVERT_TO_HF:
    HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
    
    print(f"📤 Uploading HF model to {HF_REPO_ID}...")
    
    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
        print(f"✅ Repository created/verified: {HF_REPO_ID}")
    except Exception as e:
        print(f"Note: {e}")
    
    # Create README for HF model
    hf_readme = f"""---
license: mit
language:
- en
tags:
- nanochat
- gpt
- text-generation
- conversational
pipeline_tag: text-generation
---

# {HF_REPO_NAME}

This is a NanoChat model converted to HuggingFace transformers format.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}")
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")

inputs = tokenizer("Hello, how are you?", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```
"""
    
    # Add GGUF section only if GGUF conversion is enabled
    if CONVERT_TO_GGUF:
        hf_readme += f"""
## GGUF Version

GGUF quantized versions are available at: [{HF_USERNAME}/{GGUF_REPO_NAME}](https://huggingface.co/{HF_USERNAME}/{GGUF_REPO_NAME})
"""
    
    hf_readme += """
## License

MIT License
"""
    
    readme_path = f"{HF_OUTPUT_DIR}/README.md"
    with open(readme_path, "w") as f:
        f.write(hf_readme)
    
    # Upload
    api.upload_folder(
        folder_path=HF_OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message="Upload NanoChat HF model",
    )
    
    print(f"✅ HF model uploaded!")
    print(f"🔗 https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Upload GGUF models to separate repository
if CONVERT_TO_GGUF:
    GGUF_REPO_ID = f"{HF_USERNAME}/{GGUF_REPO_NAME}"
    
    print(f"📤 Uploading GGUF models to {GGUF_REPO_ID}...")
    
    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=GGUF_REPO_ID, repo_type="model", exist_ok=True)
        print(f"✅ Repository created/verified: {GGUF_REPO_ID}")
    except Exception as e:
        print(f"Note: {e}")
    
    # Create README for GGUF repo
    all_quants = [GGUF_BASE_DTYPE] + GGUF_QUANTIZATIONS
    quant_list = "\n".join([f"- `{q}`: {model_name}-{q}.gguf" for q in all_quants])
    
    gguf_readme = f"""---
license: mit
language:
- en
tags:
- nanochat
- gguf
- llama-cpp
- text-generation
- quantized
pipeline_tag: text-generation
---

# {GGUF_REPO_NAME}

GGUF quantized versions of [{HF_USERNAME}/{HF_REPO_NAME}](https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}) for use with llama.cpp, ollama, and other GGUF-compatible inference engines.

## Available Quantizations

{quant_list}

## Usage with llama.cpp

```bash
# Download a GGUF file
huggingface-cli download {GGUF_REPO_ID} {model_name}-q4_K_M.gguf --local-dir .

# Run with llama.cpp
./llama-cli -m {model_name}-q4_K_M.gguf -p "Hello, how are you?" -n 100
```

## Usage with Ollama

Create a `Modelfile`:

```
FROM ./{model_name}-q4_K_M.gguf

TEMPLATE "{{{{.Prompt}}}}"
```

Then:

```bash
ollama create {model_name} -f Modelfile
ollama run {model_name}
```

## Quantization Details

| Quantization | Description | Use Case |
|-------------|-------------|----------|
| f16 | 16-bit float | Best quality, largest size |
| q8_0 | 8-bit quantization | Good quality, moderate size |
| q4_K_M | 4-bit K-quant (medium) | Good balance of quality and size |

## Original Model

HuggingFace format: [{HF_USERNAME}/{HF_REPO_NAME}](https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME})

## License

MIT License
"""
    
    gguf_readme_path = f"{GGUF_OUTPUT_DIR}/README.md"
    with open(gguf_readme_path, "w") as f:
        f.write(gguf_readme)
    
    # Upload all GGUF files
    api.upload_folder(
        folder_path=GGUF_OUTPUT_DIR,
        repo_id=GGUF_REPO_ID,
        repo_type="model",
        commit_message="Upload GGUF quantized models",
    )
    
    print(f"✅ GGUF models uploaded!")
    print(f"🔗 https://huggingface.co/{GGUF_REPO_ID}")

## 7️⃣ Summary

In [ ]:
print("="*60)
print("✅ CONVERSION AND UPLOAD COMPLETE!")
print("="*60)

if CONVERT_TO_HF:
    print(f"\n📦 HuggingFace Model:")
    print(f"   Local: {HF_OUTPUT_DIR}")
    print(f"   Hub:   https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}")

if CONVERT_TO_GGUF:
    print(f"\n📦 GGUF Models:")
    print(f"   Local: {GGUF_OUTPUT_DIR}")
    print(f"   Hub:   https://huggingface.co/{HF_USERNAME}/{GGUF_REPO_NAME}")
    all_quants = [GGUF_BASE_DTYPE] + GGUF_QUANTIZATIONS
    print(f"\n   Quantizations: {', '.join(all_quants)}")

print("\n" + "="*60)

## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to clean up temporary files
# import shutil
# shutil.rmtree(WORK_DIR)
# print(f"🧹 Cleaned up {WORK_DIR}")